### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [4]:
%pip install Langchain

Note: you may need to restart the kernel to use updated packages.


c:\Users\VikashMahato\GenAI_Learning\venv\Scripts\python.exe: No module named pip


In [5]:
### Open AI API Key and Open Source models--Llama3,Gemma2,mistral--Groq

import os
from dotenv import load_dotenv
load_dotenv()

import openai
groq_api_key=os.getenv("GROQ_API_KEY")


In [6]:
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

try:
    models = client.models.list()
    print(models)
    print("Groq API key is working ✅")
except Exception as e:
    print("Groq API key is NOT working ❌")
    print(e)

ModelListResponse(data=[Model(id='openai/gpt-oss-safeguard-20b', created=1761708789, object='model', owned_by='OpenAI', active=True, context_window=131072, public_apps=None, max_completion_tokens=65536), Model(id='meta-llama/llama-4-scout-17b-16e-instruct', created=1743874824, object='model', owned_by='Meta', active=True, context_window=131072, public_apps=None, max_completion_tokens=8192), Model(id='allam-2-7b', created=1737672203, object='model', owned_by='SDAIA', active=True, context_window=4096, public_apps=None, max_completion_tokens=4096), Model(id='moonshotai/kimi-k2-instruct', created=1752435491, object='model', owned_by='Moonshot AI', active=True, context_window=131072, public_apps=None, max_completion_tokens=16384), Model(id='qwen/qwen3-32b', created=1748396646, object='model', owned_by='Alibaba Cloud', active=True, context_window=131072, public_apps=None, max_completion_tokens=40960), Model(id='whisper-large-v3', created=1693721698, object='model', owned_by='OpenAI', active=

In [7]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

c:\Users\VikashMahato\GenAI_Learning\venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000269E82438B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000269E8243130>, model_name='llama-3.1-8b-instant', groq_api_key=SecretStr('**********'))

In [8]:
from langchain_core.messages import HumanMessage,SystemMessage
messages = [
    SystemMessage(content="Translate the following English to French"),
    HumanMessage(content="Hello How are you?")
]

result = model.invoke(messages)

In [9]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(result)

'Bonjour comment allez-vous ?'

In [10]:
### Using LCEL to chain the components together and create a simple app

chain = model | parser
chain.invoke(messages)

'Bonjour Comment allez-vous ?'

In [11]:
### Prompt Templates

from langchain_core.prompts import ChatPromptTemplate
generic_template = "Translate the following into {language}:"

prompt = ChatPromptTemplate.from_messages(
    [ ("system", generic_template),("user","{text}")]  
)

In [12]:
result =prompt.invoke({"language":"French","text":"Hello How are you?"})

In [13]:
result.to_messages()

[SystemMessage(content='Translate the following into French:'),
 HumanMessage(content='Hello How are you?')]

In [14]:
# Chaining together  components with LCEL 
chain = prompt | model | parser
chain.invoke({"language":"French","text":"Hello How are you?"}) 

'Bonjour comment allez-vous ?'